# Warehouse analysis: pallets, SKU physical profile, velocity and pick rates

Reads the cleaned files from `data/processed/` (`skus.parquet`, `picks.parquet`) and answers:

1. Pallet height and weight distributions
2. Pallet Ti x Hi
3. Pallet eligibility (weight < 3,000 lb and height < 8 ft)
4. SKU velocity / Pareto
5. Pallet, case and each picks per hour
6. Pallets on hand
7. Lightest and heaviest pieces and cases, volume, density, and orientation

**Before running:** set `WGT_UNIT` in the config cell. The notebook stops at the units check until you do, because a wrong weight unit shifts every eligibility result by a factor of 2.2.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 50)
C1, C2, RED, GREEN = "#4C78A8", "#F28E2B", "#E15759", "#59A14F"

project_dir = "\\wsl.localhost\Ubuntu\home\igaml\projects\warehouse-optimization"
proc = project_dir / "data" / "processed"
fig_dir = project_dir / "reports" / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

skus = pd.read_parquet(proc / "skus.parquet")
picks = pd.read_parquet(proc / "picks.parquet")
print(f"skus: {len(skus):,} rows | picks: {len(picks):,} rows")
print("pick columns:", picks.columns.tolist())

def save(fig, name):
    fig.tight_layout()
    fig.savefig(fig_dir / f"{name}.png", dpi=160, bbox_inches="tight")
    plt.show()

StopIteration: 

## 0. Configuration
Only `WGT_UNIT` is mandatory. Everything else has a safe default. `PICK_TYPE_COL`, `PICKER_COL` and `INVENTORY_FILE` depend on what exists in your raw data; section 5 and 6 explain what changes if they stay `None`.

In [ ]:
WGT_UNIT = "kg"          # REQUIRED: "lb" or "kg"
DIM_UNIT = None          # "in", "cm" or "mm". None = auto-detect from pallet footprint (verify the printout)

MAX_LB, MAX_IN = 3000, 96        # eligibility limits: 3,000 lb and 8 ft
ABC_CUTS = (0.80, 0.95)          # Pareto class boundaries (cumulative share of pick lines)

TIME_COL = "DT_START"            # pick timestamp
PICK_TYPE_COL = None             # a column holding pick type, if one exists in picks
PICK_TYPE_MAP = {}               # e.g. {"PLT": "pallet", "CSE": "case", "EA": "each"}
PICKER_COL = None                # a column identifying the picker, if one exists

INVENTORY_FILE = None            # path to an on-hand inventory file (.csv or .parquet), if one exists
INV_SKU_COL, INV_QTY_COL = "SKU", None
INV_QTY_UNIT = "each"            # "each", "case" or "pallet": the unit of the inventory quantity

## 1. SKU physical attributes
Each SKU has up to three unit levels (`UOM1`, `UOM2`, `UOM3`). Pallet attributes come from the level whose unit code is `PT`, case attributes from `CJ`, and each/piece attributes from level 1. A value of 0 means no data and becomes `NaN`.

In [ ]:
LEVELS = ["1", "2", "3"]

def level_of(df, code):
    hit = pd.DataFrame({k: df[f"UOM{k}"].astype(str).str.strip().eq(code) for k in LEVELS})
    return hit.idxmax(axis=1).where(hit.any(axis=1))

def at_level(df, stem, level):
    out = pd.Series(np.nan, index=df.index)
    for k in LEVELS:
        m = level.eq(k)
        out[m] = df.loc[m, f"{stem}{k}"]
    return out.where(out > 0)

lvl = {"each": pd.Series("1", index=skus.index),
       "case": level_of(skus, "CJ"),
       "pallet": level_of(skus, "PT")}

for name, lv in lvl.items():
    for stem in ["HGT", "WGT", "WID", "DPTH"]:
        skus[f"{stem}_{name}"] = at_level(skus, stem, lv)

q12, q23 = skus["QTY1IN2"], skus["QTY2IN3"]
lp, lc = lvl["pallet"], lvl["case"]
skus["eaches_per_case"] = q12.where(lc.eq("2"))
skus["eaches_per_pallet"] = pd.Series(np.select([lp.eq("3"), lp.eq("2")], [q12 * q23, q12], np.nan), index=skus.index)
skus["cases_per_pallet"] = q23.where(lp.eq("3"))
skus["pallet_base_units"] = pd.Series(np.select([lp.eq("3"), lp.eq("2")], [q23, q12], np.nan), index=skus.index)

print("SKUs with a pallet level :", lp.notna().sum())
print("SKUs with a case level   :", lc.notna().sum())
for name in lvl:
    print(f"{name:7s} dims+weight present:",
          skus[[f"HGT_{name}", f"WID_{name}", f"DPTH_{name}", f"WGT_{name}"]].notna().all(axis=1).sum())

## 2. Units check
Pallet footprints are usually 48 x 40 in (about 120 x 100 cm, or 1200 x 1000 mm), which identifies the dimension unit. Weight cannot be detected reliably, so use the percentiles below to set `WGT_UNIT` (loaded pallets are typically about 500-2,500 lb, or 230-1,100 kg).

In [ ]:
foot = pd.concat([skus["WID_pallet"], skus["DPTH_pallet"]]).dropna()
print("pallet footprint (WID/DPTH) median:", round(foot.median(), 1))
print(skus[["HGT_pallet", "WGT_pallet"]].describe(percentiles=[.05, .5, .95]).round(1))

if DIM_UNIT is None:
    med = foot.median()
    DIM_UNIT = "in" if 30 <= med <= 60 else "cm" if 80 <= med <= 140 else "mm" if 800 <= med <= 1400 else None
    print("DIM_UNIT auto-detected as:", DIM_UNIT, "(set it in the config cell to confirm)")
assert DIM_UNIT, "Could not detect the dimension unit; set DIM_UNIT in the config cell."
assert WGT_UNIT, "Set WGT_UNIT ('lb' or 'kg') in the config cell, using the weight percentiles printed above."

to_in = {"in": 1.0, "cm": 1 / 2.54, "mm": 1 / 25.4}[DIM_UNIT]
to_lb = {"lb": 1.0, "kg": 2.20462}[WGT_UNIT]

for name in lvl:
    skus[f"h_{name}"] = skus[f"HGT_{name}"] * to_in
    skus[f"w_{name}"] = skus[f"WID_{name}"] * to_in
    skus[f"d_{name}"] = skus[f"DPTH_{name}"] * to_in
    skus[f"lb_{name}"] = skus[f"WGT_{name}"] * to_lb
    skus[f"ft3_{name}"] = skus[f"h_{name}"] * skus[f"w_{name}"] * skus[f"d_{name}"] / 1728
    skus[f"dens_{name}"] = skus[f"lb_{name}"] / skus[f"ft3_{name}"]

# Independent cross-check of the unit and level mapping:
# pallet height should be about (layers x case height) plus the pallet base, so the ratio sits a little above 1.
ratio = (skus["h_pallet"] / (skus["LAYERS_PER_PALLET"] * skus["h_case"])).replace([np.inf, -np.inf], np.nan).dropna()
print(f"\npallet height / (layers x case height): median {ratio.median():.2f} over {len(ratio):,} SKUs (expect ~1.0-1.2)")

## 3. Pallet height distribution

In [ ]:
h = skus["h_pallet"].dropna()
print(h.describe(percentiles=[.05, .25, .5, .75, .95]).round(1))
print(f"pallet SKUs at or above {MAX_IN} in: {(h >= MAX_IN).sum():,} ({(h >= MAX_IN).mean():.1%})")

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(h, bins=40, color=C1)
ax.axvline(MAX_IN, color=RED, ls="--", label=f"{MAX_IN} in (8 ft) limit")
ax.set(xlabel="pallet height (in)", ylabel="SKUs", title="Pallet height distribution (SKU master)")
ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False)
save(fig, "pallet_height_distribution")

## 4. Pallet weight distribution

In [ ]:
wt = skus["lb_pallet"].dropna()
print(wt.describe(percentiles=[.05, .25, .5, .75, .95]).round(1))
print(f"pallet SKUs at or above {MAX_LB:,} lb: {(wt >= MAX_LB).sum():,} ({(wt >= MAX_LB).mean():.1%})")
print("heaviest 5 (check for data errors before trusting these):")
print(skus.loc[wt.nlargest(5).index, ["SKU", "lb_pallet"]].to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(wt, bins=40, color=C1)
ax.axvline(MAX_LB, color=RED, ls="--", label=f"{MAX_LB:,} lb limit")
ax.set(xlabel="pallet weight (lb)", ylabel="SKUs", title="Pallet weight distribution (SKU master)")
ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False)
save(fig, "pallet_weight_distribution")

## 5. Pallet Ti x Hi
Ti is cases per layer and Hi is layers per pallet, so Ti x Hi is the pallet quantity in the unit one level below the pallet. It should equal the master's own pallet quantity, which makes it a built-in validation.

In [ ]:
skus["Ti"] = skus["CASES_PER_LAYER"]
skus["Hi"] = skus["LAYERS_PER_PALLET"]
skus["TixHi"] = skus["Ti"] * skus["Hi"]

ok = skus.dropna(subset=["TixHi", "pallet_base_units"])
match = ok["TixHi"] == ok["pallet_base_units"]
print(f"SKUs with Ti, Hi and a master pallet quantity: {len(ok):,}")
print(f"Ti x Hi equals the master pallet quantity for {match.mean():.1%} of them")
print("\nTi x Hi distribution:")
print(skus["TixHi"].describe(percentiles=[.05, .5, .95]).round(1))
print("\nmost common Ti / Hi combinations:")
print(skus.groupby(["Ti", "Hi"]).size().sort_values(ascending=False).head(10).rename("SKUs"))
print("\nexamples where Ti x Hi disagrees with the master:")
print(ok.loc[~match, ["SKU", "Ti", "Hi", "TixHi", "pallet_base_units"]].head(10).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(skus["TixHi"].dropna(), bins=40, color=C1)
axes[0].set(xlabel="Ti x Hi (units per pallet)", ylabel="SKUs", title="Ti x Hi distribution")
hi = skus["Hi"].dropna().astype(int).value_counts().sort_index()
axes[1].bar(hi.index.astype(str), hi.values, color=C2)
axes[1].set(xlabel="Hi (layers per pallet)", ylabel="SKUs", title="Layers per pallet")
for a in axes: a.spines[["top", "right"]].set_visible(False)
save(fig, "pallet_tixhi")

## 6. Pallet eligibility (< 3,000 lb and < 8 ft)
Evaluated on the SKU master's full-pallet weight and height. SKUs without both values are reported separately instead of being counted as eligible or ineligible. Results are shown per SKU and weighted by pick lines.

In [ ]:
p = skus[skus["lb_pallet"].notna() | skus["h_pallet"].notna() | lvl["pallet"].notna()].copy()
no_data = p["lb_pallet"].isna() | p["h_pallet"].isna()
w_ok, h_ok = p["lb_pallet"] < MAX_LB, p["h_pallet"] < MAX_IN

p["status"] = np.select(
    [no_data, w_ok & h_ok, ~w_ok & h_ok, w_ok & ~h_ok],
    ["no weight/height data", "eligible", "fails weight only", "fails height only"],
    default="fails both")

by_sku = p["status"].value_counts().rename("SKUs").to_frame().rename_axis("status")
by_sku["pct"] = by_sku["SKUs"] / len(p) * 100
print("By SKU (pallet-configured SKUs only):")
print(by_sku)

status_map = p.drop_duplicates("SKU").set_index("SKU")["status"]
fallback = pd.Series(np.where(picks["SKU"].isin(skus["SKU"]), "no pallet unit", "not in master"), index=picks.index)
line_status = picks["SKU"].map(status_map).fillna(fallback)
by_line = line_status.value_counts().rename("pick_lines").to_frame().rename_axis("status")
by_line["pct"] = by_line["pick_lines"] / len(picks) * 100
print("\nBy pick lines (all lines):")
print(by_line)

if "was_repaired" in p.columns:
    print("\nStatus mix, repaired vs untouched SKU rows (a large gap means inferred values drive the result):")
    print(p.groupby("was_repaired")["status"].value_counts(normalize=True).round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, tbl, title in [(axes[0], by_sku["pct"], "% of pallet SKUs"), (axes[1], by_line["pct"], "% of pick lines")]:
    ax.barh(tbl.index[::-1], tbl.values[::-1], color=C1)
    for y, v in enumerate(tbl.values[::-1]): ax.text(v, y, f" {v:.1f}%", va="center", fontsize=9)
    ax.set_title(f"Eligibility: {title}"); ax.spines[["top", "right"]].set_visible(False)
    ax.set_xlim(0, max(tbl.values) * 1.15)
save(fig, "pallet_eligibility")

## 7. SKU velocity / Pareto
Velocity is the number of pick lines with a nonzero `ACT_QTY` (lines are what drives slotting and travel). Pallet-equivalents moved is shown alongside for the SKUs where a pallet quantity is known.

In [ ]:
picked = picks[picks["ACT_QTY"] > 0]
vel = (picked.groupby("SKU").agg(lines=("SKU", "size"), units=("ACT_QTY", "sum"))
       .sort_values("lines", ascending=False))
total_lines = vel["lines"].sum()
vel["cum_pct"] = vel["lines"].cumsum() / total_lines
prev = vel["cum_pct"] - vel["lines"] / total_lines
vel["abc"] = np.select([prev < ABC_CUTS[0], prev < ABC_CUTS[1]], ["A", "B"], "C")
vel = vel.join(skus.drop_duplicates("SKU").set_index("SKU")[["eaches_per_pallet"]])
vel["pallets_moved"] = vel["units"] / vel["eaches_per_pallet"]

summary = vel.groupby("abc").agg(SKUs=("lines", "size"), lines=("lines", "sum"), units=("units", "sum"))
summary["pct_SKUs"] = summary["SKUs"] / len(vel) * 100
summary["pct_lines"] = summary["lines"] / total_lines * 100
print(summary)
n80 = int((prev < ABC_CUTS[0]).sum())
print(f"\n{n80:,} SKUs ({n80 / len(vel):.1%} of picked SKUs) account for {ABC_CUTS[0]:.0%} of pick lines")
print("\ntop 10 by pick lines:")
print(vel.head(10)[["lines", "units", "pallets_moved", "abc"]])

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(1, len(vel) + 1) / len(vel) * 100
ax.plot(x, vel["cum_pct"].values * 100, color=C1, lw=2)
for c in ABC_CUTS: ax.axhline(c * 100, color="grey", ls=":", lw=1)
ax.axvline(n80 / len(vel) * 100, color=RED, ls="--", label=f"{n80 / len(vel):.1%} of SKUs = {ABC_CUTS[0]:.0%} of lines")
ax.set(xlabel="% of picked SKUs (ranked by pick lines)", ylabel="cumulative % of pick lines", title="SKU velocity Pareto")
ax.legend(frameon=False, loc="lower right"); ax.spines[["top", "right"]].set_visible(False)
save(fig, "sku_pareto")

## 8. Pick rates per hour: pallet, case, each
**Pick type.** If `PICK_TYPE_COL` is set, it is used. Otherwise pick type is inferred from `ACT_QTY` against each SKU's pack sizes, assuming `ACT_QTY` is counted in eaches: a quantity that is a multiple of the pallet quantity is a pallet pick, a multiple of the case quantity is a case pick, and anything else is an each pick. This is an assumption, and a case-multiple quantity can also be a legitimate each pick, so a real pick-type column is better. The cell below lists candidate columns.

**Rate.** Without a picker column, the rate is picks per active clock hour (the whole operation), not per labor hour. Set `PICKER_COL` to also get picks per picker-hour.

In [ ]:
# candidate pick-type / picker columns: low-cardinality text columns in picks
for c in picks.columns:
    n = picks[c].nunique(dropna=True)
    if picks[c].dtype == object and n <= 25:
        print(f"{c:20s} {n:3d} values: {picks[c].dropna().unique()[:8].tolist()}")

In [ ]:
extra = [c for c in [PICK_TYPE_COL, PICKER_COL] if c]
attr = skus.drop_duplicates("SKU")[["SKU", "eaches_per_case", "eaches_per_pallet"]]
pk = picks[["SKU", TIME_COL, "ACT_QTY", "REQ_QTY"] + extra].merge(attr, on="SKU", how="left")
q = pk["ACT_QTY"].astype(float)

if PICK_TYPE_COL:
    pk["pick_type"] = pk[PICK_TYPE_COL].map(PICK_TYPE_MAP).fillna("unclassified")
else:
    epp, epc = pk["eaches_per_pallet"], pk["eaches_per_case"]
    with np.errstate(divide="ignore", invalid="ignore"):
        pal = (epp > 0) & (q >= epp) & (q % epp == 0)
        cas = ~pal & (epc > 0) & (q >= epc) & (q % epc == 0)
    no_pack = epp.isna() & epc.isna()
    pk["pick_type"] = np.select([q <= 0, no_pack, pal, cas], ["none", "unclassified", "pallet", "case"], default="each")
pk.loc[q <= 0, "pick_type"] = "none"

pk["ts"] = pd.to_datetime(pk[TIME_COL], errors="coerce")
print("unparseable timestamps:", pk["ts"].isna().sum())
print("date range:", pk["ts"].min(), "to", pk["ts"].max())
mix = pk["pick_type"].value_counts().rename("lines").to_frame()
mix["pct"] = mix["lines"] / len(pk) * 100
print(mix)

In [ ]:
done = pk[pk["pick_type"].isin(["pallet", "case", "each"])].copy()
done["hour"] = done["ts"].dt.floor("h")
hourly = done.groupby(["hour", "pick_type"]).size().unstack(fill_value=0)
for t in ["pallet", "case", "each"]:
    if t not in hourly: hourly[t] = 0
hourly = hourly[["pallet", "case", "each"]]

stats = hourly.agg(["mean", "median", lambda s: s.quantile(.95), "max"]).T
stats.columns = ["mean", "median", "p95", "max"]
print(f"Picks per active clock hour ({len(hourly):,} active hours):")
print(stats)
print("\nPeak hour (all types):", hourly.sum(axis=1).idxmax(), "with", int(hourly.sum(axis=1).max()), "picks")

if PICKER_COL:
    ph = done.groupby(["hour", PICKER_COL, "pick_type"]).size().unstack(fill_value=0)
    print("\nPicks per active picker-hour (mean):")
    print(ph.mean().round(2))

prof = hourly.groupby(hourly.index.hour).mean()
fig, ax = plt.subplots(figsize=(10, 4.5))
for t, col in [("pallet", RED), ("case", C1), ("each", GREEN)]:
    ax.plot(prof.index, prof[t], marker="o", label=t, color=col)
ax.set(xlabel="hour of day", ylabel="mean picks per hour", title="Picks per hour by hour of day")
ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False)
save(fig, "picks_per_hour_by_type")

## 9. Pallets on hand
Picks are outbound only, so on-hand stock cannot be derived from them. This section needs an inventory snapshot. If `INVENTORY_FILE` is `None` it only reports that.

In [ ]:
if INVENTORY_FILE is None:
    print("No inventory file configured: pallets on hand cannot be derived from outbound picks alone.")
    print("Look in the raw data folder for an inventory/stock/on-hand file, then set INVENTORY_FILE and INV_QTY_COL.")
else:
    f = Path(INVENTORY_FILE)
    inv = pd.read_csv(f) if f.suffix == ".csv" else pd.read_parquet(f)
    assert INV_QTY_COL, "Set INV_QTY_COL to the quantity column in the inventory file."
    per = {"each": "eaches_per_pallet", "case": "cases_per_pallet"}.get(INV_QTY_UNIT)
    inv = inv.rename(columns={INV_SKU_COL: "SKU"})
    if per:
        inv = inv.merge(skus.drop_duplicates("SKU")[["SKU", per]], on="SKU", how="left")
        inv["pallets"] = inv[INV_QTY_COL] / inv[per]
    else:
        inv["pallets"] = inv[INV_QTY_COL]
    print(f"inventory rows: {len(inv):,} | rows with no pallet quantity in the master: {inv['pallets'].isna().sum():,}")
    print(f"full-pallet equivalents on hand: {inv['pallets'].sum():,.0f}")
    print(f"minimum pallet positions (partials round up per SKU): {np.ceil(inv.groupby('SKU')['pallets'].sum()).sum():,.0f}")

## 10. Lightest and heaviest pieces and cases
Extremes are where data errors hide, so each list shows whether the SKU row was repaired or flagged as suspect (when those columns exist). Check a few against the source before drawing conclusions from them.

In [ ]:
flags = [c for c in ["was_repaired", "flag_suspect"] if c in skus.columns]

def extremes(col, label, n=10):
    d = skus.dropna(subset=[col])
    print(f"--- {label}: {len(d):,} SKUs with data ---")
    print("lightest:"); print(d.nsmallest(n, col)[["SKU", col] + flags].to_string(index=False))
    print("heaviest:"); print(d.nlargest(n, col)[["SKU", col] + flags].to_string(index=False))
    print()

extremes("lb_each", "piece (each) weight, lb")
extremes("lb_case", "case weight, lb")

## 11. Which SKUs occupy the most volume?
Without inventory, occupancy is measured as volume moved: units picked x volume per unit. Volume per unit uses each-level dimensions, falling back to case volume divided by units per case. A SKU's physical size per unit and per case is ranked too.

In [ ]:
s1 = skus.drop_duplicates("SKU").set_index("SKU")
vol_each = s1["ft3_each"].fillna(s1["ft3_case"] / s1["eaches_per_case"])
vm = picked.groupby("SKU")["ACT_QTY"].sum().to_frame("units")
vm["in3_per_unit"] = vol_each.reindex(vm.index) * 1728
vm["ft3_moved"] = vm["units"] * vm["in3_per_unit"] / 1728
known = vm["ft3_moved"].notna()
print(f"picked units with a known volume: {vm.loc[known, 'units'].sum() / vm['units'].sum():.1%}")
top = vm.sort_values("ft3_moved", ascending=False).head(15)
top["share_of_volume"] = top["ft3_moved"] / vm["ft3_moved"].sum() * 100
print("\ntop 15 SKUs by cubic feet picked:")
print(top)

print("\nlargest case volume (ft3):")
print(skus.nlargest(10, "ft3_case")[["SKU", "h_case", "w_case", "d_case", "ft3_case"] + flags].to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
t = top.iloc[::-1]
ax.barh(t.index.astype(str), t["ft3_moved"], color=C1)
ax.set(xlabel="cubic feet picked", title="Top 15 SKUs by volume moved")
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:,.0f}")); ax.spines[["top", "right"]].set_visible(False)
save(fig, "sku_volume_moved")

## 12. Which SKUs are the densest (weight / volume)?
Density is lb per cubic foot. As a sanity anchor, water is about 62 lb/ft3, so a case density far above ~500 or near 0 is usually a unit or data error, not a real product.

In [ ]:
d = skus.dropna(subset=["dens_case"])
print(d["dens_case"].describe(percentiles=[.05, .5, .95]).round(1))
print("\ndensest cases:")
print(d.nlargest(10, "dens_case")[["SKU", "lb_case", "ft3_case", "dens_case"] + flags].to_string(index=False))
print("\nleast dense cases:")
print(d.nsmallest(10, "dens_case")[["SKU", "lb_case", "ft3_case", "dens_case"] + flags].to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(np.log10(d["dens_case"]), bins=40, color=C1)
ax.set(xlabel="case density, log10(lb/ft3)", ylabel="SKUs", title="Case density distribution")
ax.spines[["top", "right"]].set_visible(False)
save(fig, "case_density")

## 13. Is case height the largest case dimension?
**Why it is useful to know.**
- **Pallet height and eligibility.** Pallet height is roughly Hi x case height plus the base. A case that stands on its tallest side builds the tallest possible pallet. If the product allows rotating it, lying it on its largest face can bring a pallet under the 8 ft limit or fit another layer.
- **Stability.** Tall, narrow cases are more likely to tip and shift on a pallet, so they often cap Hi lower than weight alone would.
- **Slotting.** Tall-case SKUs need more vertical clearance in a bay, so they constrain which locations and rack levels they can use.
- **Handling.** Orientation (and "this side up" limits) decides whether a case can be turned to reduce height, which only matters for SKUs where height dominates.

In [ ]:
c = skus.dropna(subset=["h_case", "w_case", "d_case"]).copy()
other_max = c[["w_case", "d_case"]].max(axis=1)
c["height_largest"] = c["h_case"] > other_max
c["height_tied"] = (c["h_case"] == other_max)
c["tall_ratio"] = c["h_case"] / other_max

n = len(c)
print(f"SKUs with full case dimensions: {n:,}")
print(f"height is strictly the largest dimension: {c['height_largest'].sum():,} ({c['height_largest'].mean():.1%})")
print(f"height ties for largest: {c['height_tied'].sum():,}")

lines_per_sku = picked.groupby("SKU").size()
c["lines"] = c["SKU"].map(lines_per_sku).fillna(0)
print(f"share of pick lines on height-largest SKUs: {c.loc[c['height_largest'], 'lines'].sum() / c['lines'].sum():.1%}")

if "h_pallet" in c:
    both = c.dropna(subset=["h_pallet"])
    print(f"\nof pallet SKUs with height-largest cases: {(both.loc[both['height_largest'], 'h_pallet'] >= MAX_IN).sum():,} fail the {MAX_IN} in limit")

print("\ntallest relative to their other dimensions:")
print(c.nlargest(10, "tall_ratio")[["SKU", "h_case", "w_case", "d_case", "tall_ratio", "lines"]].to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(c["tall_ratio"].clip(upper=3), bins=40, color=C1)
ax.axvline(1, color=RED, ls="--", label="height = largest other dimension")
ax.set(xlabel="case height / larger of width and depth (clipped at 3)", ylabel="SKUs", title="How tall are cases relative to their footprint?")
ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False)
save(fig, "case_height_ratio")

## 14. Save the enriched SKU table and note the limits

In [ ]:
skus.to_parquet(proc / "sku_attributes.parquet", index=False)
print("saved", proc / "sku_attributes.parquet")
print("figures in", fig_dir)

### Assumptions and limits
- **Units** for dimensions and weight come from the config cell. Dimension units are auto-detected from pallet footprint unless set.
- **Pallet attributes are full-pallet master values.** Partial and mixed pallets in the building can be lighter or shorter.
- **Pick type is inferred** unless `PICK_TYPE_COL` is set (assumes `ACT_QTY` is in eaches).
- **Picks per hour is per active clock hour**, not per labor hour, unless `PICKER_COL` is set.
- **Pallets on hand** requires an inventory file; it cannot be derived from picks.
- **About 1.4% of pick lines have no SKU master record**, and 16 SKU master rows were excluded as unrepairable. Attribute-based results cover the rest. The 140 repaired SKU rows are inferred, not verified.
- **Whether pallet height includes the pallet base** is not known from the data; confirm it, because borderline SKUs near 96 in depend on it.